# Block 2 · Teil C: Hybride Verschlüsselung (Partnerarbeit)

Jetzt arbeitet ihr mit einer echten, geprüften Kryptografie-Bibliothek (`cryptography`). Einer ist **Bob** (Empfänger), der andere **Alice** (Absenderin).
**Beide** öffnen dieses Notebook auf ihrem eigenen Gerät. In jeder Überschrift steht, wer die Zellen ausführt.

Schlüssel und Pakete tauscht ihr als **Text über den Chat** eurer Schulplattform aus: Ausgabe markieren, kopieren, im Chat einfügen.
Allein? Dann spielst du beide Rollen in diesem Notebook.

**So arbeitest du**

- Zelle anklicken, dann **Umschalt + Enter**: Die Zelle wird ausgeführt. Arbeite von oben nach unten.
- Deine Antworten schreibst du auf das **Aufgabenblatt**. Die Überschriften hier tragen dieselben Aufgabennummern.
- **Vermute zuerst:** Schreib deine Vermutung auf, *bevor* du die Zelle ausführst. Danach vergleichen.
- Werte, die du ändern darfst, stehen in GROSSBUCHSTABEN oben in der Zelle.
- Seltsame Fehlermeldung? Menü **Kernel → Kernel neu starten** (englisch: *Restart Kernel*), dann alle Zellen von oben erneut ausführen.
- Dieses Notebook speichert Änderungen nur in **diesem Browser**. Auf Schulrechnern können sie nach dem Abmelden weg sein. Wichtig ist, was auf deinem Aufgabenblatt steht.

## Vorbereitung [beide]
Die erste Zelle lädt die Bibliothek (beim ersten Mal einige Sekunden), die zweite stellt Hilfsfunktionen bereit.

In [ ]:
%pip install -q cryptography

In [ ]:
import base64, hashlib, os, textwrap, time
from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

RSA_BIT = 3072    # Empfehlung des BSI: mindestens 3000 Bit
OAEP = padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
MAGIC = b"KRYPT1"


def lade_oeffentlich(pem_text):
    """Öffentlichen Schlüssel aus eingefügtem Text laden (mit verständlicher Fehlermeldung)."""
    if "PRIVATE KEY" in pem_text:
        print("STOPP: Das ist ein PRIVATER Schlüssel. Den gibt man niemals weiter!")
        return None
    try:
        return serialization.load_pem_public_key(pem_text.strip().encode())
    except ValueError:
        print("Das ist kein vollständiger öffentlicher Schlüssel. Hast du alles kopiert, auch die Zeilen BEGIN und END?")
        return None


def fingerprint(pem_text):
    """SHA-256 des Schlüssels, gekürzt auf 32 Hex-Zeichen, in Vierergruppen."""
    schluessel = lade_oeffentlich(pem_text)
    if schluessel is None:
        return "–"
    der = schluessel.public_bytes(serialization.Encoding.DER, serialization.PublicFormat.SubjectPublicKeyInfo)
    h = hashlib.sha256(der).hexdigest()[:32].upper()
    return " ".join(h[i:i + 4] for i in range(0, 32, 4))


def paket_packen(schluessel_geheim, nonce, geheimtext):
    """Fasst alles zu einem Text zusammen, den man im Chat verschicken kann."""
    roh = MAGIC + len(schluessel_geheim).to_bytes(2, "big") + schluessel_geheim + nonce + geheimtext
    b64 = base64.b64encode(roh).decode()
    return "-----BEGIN KRYPT-PAKET-----\n" + "\n".join(textwrap.wrap(b64, 64)) + "\n-----END KRYPT-PAKET-----"


def paket_entpacken(text):
    """Gegenstück zu paket_packen. Gibt (schluessel_geheim, nonce, geheimtext) zurück oder None."""
    zeilen = [z.strip() for z in text.strip().splitlines() if z.strip() and "KRYPT-PAKET" not in z]
    try:
        roh = base64.b64decode("".join(zeilen), validate=True)
    except ValueError:
        roh = b""
    if not roh.startswith(MAGIC):
        print("Das ist kein vollständiges Paket. Hast du alles kopiert, auch die Zeilen BEGIN und END?")
        return None
    i = len(MAGIC)
    lk = int.from_bytes(roh[i:i + 2], "big")
    i += 2
    return roh[i:i + lk], roh[i + lk:i + lk + 12], roh[i + lk + 12:]


def lade_privat(pfad, passwort):
    try:
        with open(pfad, "rb") as f:
            return serialization.load_pem_private_key(f.read(), password=passwort.encode())
    except FileNotFoundError:
        print(f"{pfad} fehlt. Wurde die Seite neu geladen? Dann C1 erneut ausführen und Alice den neuen Schlüssel schicken.")
    except (ValueError, TypeError):
        print("Falsches Passwort: Der private Schlüssel lässt sich nicht öffnen.")
    return None


def verschluesseln_still(text, pem_text):
    """Dasselbe wie C3, nur ohne Ausgaben."""
    schluessel = lade_oeffentlich(pem_text)
    k = AESGCM.generate_key(bit_length=256)
    nonce = os.urandom(12)
    return paket_packen(schluessel.encrypt(k, OAEP), nonce, AESGCM(k).encrypt(nonce, text.encode(), None))


def ein_oeffentlicher_schluessel():
    """Für C5: Bobs Schlüssel aus C2 (Alice) oder den eigenen aus C1 (Bob)."""
    for name in ("BOB_OEFFENTLICH", "oeffentlich_pem"):
        wert = globals().get(name, "")
        if "BEGIN PUBLIC KEY" in wert:
            return lade_oeffentlich(wert)
    print("Erzeuge einen Test-Schlüssel ...")
    return rsa.generate_private_key(public_exponent=65537, key_size=RSA_BIT).public_key()


print("Bereit.")

## C1 Bob erzeugt sein Schlüsselpaar [Bob]

Trage ein Passwort ein, das den privaten Schlüssel schützen soll. **Nimm kein echtes Passwort!**
Echte Programme fragen das Passwort verdeckt ab und speichern es nirgends, so wie `ssh-keygen` in Teil D. Hier steht es nur zur Vereinfachung in der Zelle.

Das Erzeugen dauert im Browser einige Sekunden.

In [ ]:
PASSWORT = "uebung-123"    # ← ändern, aber kein echtes Passwort

print(f"Erzeuge RSA-Schlüsselpaar mit {RSA_BIT} Bit ...")
start = time.perf_counter()
privat = rsa.generate_private_key(public_exponent=65537, key_size=RSA_BIT)
print(f"Fertig nach {time.perf_counter() - start:.1f} s.")

# Privaten Schlüssel mit dem Passwort verschlüsselt als Datei speichern
with open("bob_privat.pem", "wb") as f:
    f.write(privat.private_bytes(serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                                 serialization.BestAvailableEncryption(PASSWORT.encode())))
oeffentlich_pem = privat.public_key().public_bytes(serialization.Encoding.PEM,
                                                   serialization.PublicFormat.SubjectPublicKeyInfo).decode()
del privat    # ab jetzt nur noch aus der Datei, mit Passwort

print("bob_privat.pem gespeichert (mit Passwort geschützt) → bleibt bei dir")
print()
print("Dein ÖFFENTLICHER Schlüssel. Komplett markieren (mit BEGIN und END), kopieren und Alice im Chat schicken:")
print()
print(oeffentlich_pem)
print("Fingerprint:", fingerprint(oeffentlich_pem))

➜ Beantworte **C1**.

## C2 Fingerprint vergleichen [Alice]

Füge Bobs öffentlichen Schlüssel aus dem Chat zwischen die dreifachen Anführungszeichen ein und führe die Zelle aus.
Bob liest seinen Fingerprint **mündlich** vor, du vergleichst.

In [ ]:
BOB_OEFFENTLICH = """
hier den Schlüssel aus dem Chat einfügen, mit den Zeilen BEGIN und END
"""

print("Fingerprint:", fingerprint(BOB_OEFFENTLICH))

➜ Beantworte **C2**.

## C3 Alice verschlüsselt [Alice]

Die Zelle zeigt die Schritte der hybriden Verschlüsselung einzeln. Ändere die Nachricht nach Belieben.

In [ ]:
NACHRICHT = "Das WLAN-Passwort beim Kunden lautet: Sonnenblume-42"

bob = lade_oeffentlich(BOB_OEFFENTLICH)
assert bob, "Zuerst in C2 Bobs öffentlichen Schlüssel einfügen und die Zelle ausführen."
klartext = NACHRICHT.encode()

# 1. Zufälligen Sitzungsschlüssel für AES-256 erzeugen
sitzungsschluessel = AESGCM.generate_key(bit_length=256)
print(f"1. Sitzungsschlüssel erzeugt: {len(sitzungsschluessel)} Byte")

# 2. Nachricht mit AES-GCM verschlüsseln (schnell, beliebig lang)
nonce = os.urandom(12)
geheimtext = AESGCM(sitzungsschluessel).encrypt(nonce, klartext, None)
print(f"2. Nachricht mit AES-GCM verschlüsselt: {len(klartext)} Byte → {len(geheimtext)} Byte")

# 3. Nur den kleinen Sitzungsschlüssel mit Bobs öffentlichem Schlüssel verschlüsseln (RSA-OAEP)
schluessel_geheim = bob.encrypt(sitzungsschluessel, OAEP)
print(f"3. Sitzungsschlüssel mit RSA-OAEP verschlüsselt: {len(schluessel_geheim)} Byte")

# 4. Alles zu einem Paket zusammenfassen
paket = paket_packen(schluessel_geheim, nonce, geheimtext)
print("4. Paket fertig. Komplett markieren (mit BEGIN und END), kopieren und Bob im Chat schicken:")
print()
print(paket)

➜ Beantworte **C3**.

## C4 Bob entschlüsselt [Bob]

Füge das Paket von Alice ein. Zum Öffnen des privaten Schlüssels brauchst du dein Passwort aus C1.

In [ ]:
PAKET = """
hier das Paket von Alice einfügen, mit den Zeilen BEGIN und END
"""
PASSWORT = "uebung-123"    # dasselbe Passwort wie in C1

privat = lade_privat("bob_privat.pem", PASSWORT)
teile = paket_entpacken(PAKET)
if privat and teile:
    schluessel_geheim, nonce, geheimtext = teile

    # 1. Sitzungsschlüssel mit dem PRIVATEN Schlüssel zurückgewinnen
    try:
        sitzungsschluessel = privat.decrypt(schluessel_geheim, OAEP)
        print(f"1. Sitzungsschlüssel mit dem privaten Schlüssel entschlüsselt: {len(sitzungsschluessel)} Byte")
    except ValueError:
        sitzungsschluessel = None
        print("1. FEHLER: Das Paket wurde nicht für deinen Schlüssel verschlüsselt.")

    # 2. Nachricht mit AES-GCM entschlüsseln; dabei wird auch geprüft, ob sie verändert wurde
    if sitzungsschluessel:
        try:
            klartext = AESGCM(sitzungsschluessel).decrypt(nonce, geheimtext, None)
            print("2. Nachricht entschlüsselt und auf Veränderung geprüft: in Ordnung")
            print()
            print("Nachricht:", klartext.decode())
        except InvalidTag:
            print("2. FEHLER: Das Paket wurde verändert oder ist beschädigt.")

➜ Beantworte **C4**. Probiere auch ein falsches Passwort aus.

## C5 Dann eben alles mit RSA? [beide]

„Dann verschlüsseln wir die 500-MB-Sicherung einfach komplett mit RSA.“ Die Zelle versucht, Daten der Größe `GROESSE` direkt mit RSA zu verschlüsseln.

**Vermute zuerst:** Wie viele Byte schafft RSA mit 3072 Bit höchstens am Stück?
Ändere danach `GROESSE` so lange, bis du die Grenze auf das Byte genau gefunden hast.

In [ ]:
GROESSE = 2_000_000    # Byte; 2 MB entsprechen etwa einem Handyfoto

schluessel = ein_oeffentlicher_schluessel()
try:
    schluessel.encrypt(os.urandom(GROESSE), OAEP)
    print(f"{GROESSE} Byte: hat geklappt.")
except ValueError:
    print(f"{GROESSE} Byte: FEHLER, zu groß für RSA-OAEP mit {schluessel.key_size} Bit.")

Jetzt der Geschwindigkeitsvergleich: 1 MB mit AES-GCM gegen 1 MB mit RSA (30 Blöcke gemessen, auf 1 MB hochgerechnet). Dauert einige Sekunden.

**Vermute zuerst:** Wie viel langsamer ist RSA: 10-mal, 100-mal oder 1000-mal?

In [ ]:
daten = os.urandom(1_000_000)
k = AESGCM.generate_key(bit_length=256)
nonce = os.urandom(12)
start = time.perf_counter()
AESGCM(k).decrypt(nonce, AESGCM(k).encrypt(nonce, daten, None), None)
t_aes = time.perf_counter() - start

print(f"Erzeuge Test-Schlüsselpaar ({RSA_BIT} Bit) ...")
test = rsa.generate_private_key(public_exponent=65537, key_size=RSA_BIT)
block = RSA_BIT // 8 - 66                  # 318 Byte pro Block
probe = daten[:30 * block]                 # 30 Blöcke messen, dann hochrechnen
start = time.perf_counter()
for i in range(0, len(probe), block):
    test.decrypt(test.public_key().encrypt(probe[i:i + block], OAEP), OAEP)
t_rsa = (time.perf_counter() - start) * len(daten) / len(probe)

print()
print("1 MB verschlüsseln und wieder entschlüsseln:")
print(f"  AES-256-GCM: {t_aes * 1000:10.1f} ms")
print(f"  RSA-{RSA_BIT}:   {t_rsa * 1000:10.1f} ms   (hochgerechnet)")
print(f"  RSA ist hier etwa {t_rsa / t_aes:,.0f}-mal langsamer.".replace(",", "."))

➜ Beantworte **C5**.

## C6 Zweimal dasselbe verschlüsseln [Alice]

Die Zelle verschlüsselt deine Nachricht aus C3 zweimal für Bob und vergleicht die SHA-256-Werte der beiden Pakete (wie in Block 1).

**Vermute zuerst:** Sind die beiden Hashwerte gleich oder verschieden?

In [ ]:
paket1 = verschluesseln_still(NACHRICHT, BOB_OEFFENTLICH)
paket2 = verschluesseln_still(NACHRICHT, BOB_OEFFENTLICH)
print("SHA-256 Paket 1:", hashlib.sha256(paket1.encode()).hexdigest())
print("SHA-256 Paket 2:", hashlib.sha256(paket2.encode()).hexdigest())

➜ Beantworte **C6**.

## Für Schnelle: F3 Ein Bit verändert [beide]

Ein Angreifer verändert unterwegs **ein einziges Bit** im verschlüsselten Teil des Pakets. Er kennt den Sitzungsschlüssel nicht.

**Vermute zuerst:** Was passiert beim Entschlüsseln: eine leicht veränderte Nachricht, Zeichensalat oder eine Fehlermeldung?

In [ ]:
manipuliert = bytearray(geheimtext)
manipuliert[0] ^= 1    # ein einziges Bit umdrehen

try:
    print(AESGCM(sitzungsschluessel).decrypt(nonce, bytes(manipuliert), None).decode())
except InvalidTag:
    print("FEHLER: Die Prüfung von AES-GCM ist fehlgeschlagen. Die Nachricht wird nicht ausgegeben.")

➜ Beantworte **F3**.